# Eigenvector Direction Check

Chapter 6: Geometric Transformation and Eigen Decomposition · *Mathematics for AI and Machine Learning* by Fuheng Wu

Test candidate directions and see when Av stays collinear with v.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-6/eigenvector-check/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.set_printoptions(precision=4)

# The demo's default matrix and probe direction (40 degrees)
A = torch.tensor([[2.1, 0.7],
                  [0.3, 1.4]], dtype=torch.float64)
theta = math.radians(40)


def collinearity(A, v):
    """|sin| of the angle between v and Av: 0 means Av is a multiple of v."""
    Av = A @ v
    cross = v[0] * Av[1] - v[1] * Av[0]
    return (cross.abs() / (v.norm() * Av.norm())).item()


v = torch.tensor([math.cos(theta), math.sin(theta)], dtype=torch.float64)
print("probe v =", v.tolist(), " A v =", (A @ v).tolist())
print("|sin angle(v, Av)| =", collinearity(A, v), "-> not an eigenvector")

# ── Eigenpairs: directions where A only stretches ───────────────────────────
evals, evecs = torch.linalg.eig(A)
evals, evecs = evals.real, evecs.real              # real for this matrix
for lam, q in zip(evals, evecs.T):
    print(f"\nlambda = {lam.item():.4f}, v = {q.tolist()}")
    print("  A v - lambda v =", (A @ q - lam * q).tolist())
    print("  |sin angle(v, Av)| =", collinearity(A, q))

# trace = sum of eigenvalues, det = product
print("\ntrace(A) =", A.trace().item(), " sum of eigenvalues =", evals.sum().item())
print("det(A)   =", torch.linalg.det(A).item(), " product of eigenvalues =", evals.prod().item())

# ── Scan all directions: collinearity hits 0 exactly at the eigenvectors ────
angles = torch.arange(0, 180, 0.5, dtype=torch.float64)
scores = torch.tensor([collinearity(A, torch.tensor([math.cos(math.radians(a)), math.sin(math.radians(a))],
                                                    dtype=torch.float64)) for a in angles])
best = angles[scores.argsort()[:2]].sort().values
print("\ndirections (deg, 0-180) where Av is parallel to v:", best.tolist())
print("eigenvector angles:", sorted(round(math.degrees(math.atan2(q[1], q[0])) % 180, 1) for q in evecs.T))

probe v = [0.766044443118978, 0.6427876096865393]  A v = [2.058644657330431, 1.1297159864968482]
|sin angle(v, Av)| = 0.1949786900378841 -> not an eigenvector

lambda = 2.3266, v = [0.9513817894361813, 0.30801410800352874]
  A v - lambda v = [0.0, -1.1102230246251565e-16]
  |sin angle(v, Av)| = 4.7718112337630266e-17

lambda = 1.1734, v = [-0.6027678380339612, 0.797916620601341]
  A v - lambda v = [0.0, 1.1102230246251565e-16]
  |sin angle(v, Av)| = 0.0

trace(A) = 3.5  sum of eigenvalues = 3.5
det(A)   = 2.7299999999999995  product of eigenvalues = 2.73

directions (deg, 0-180) where Av is parallel to v: [18.0, 127.0]
eigenvector angles: [17.9, 127.1]
